# Agente para transformação de texto em sql para analise

In [0]:
# Instalação dos pacotes necessários

%pip install --upgrade databricks-agents mlflow langchain langchain-community databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
# Imports necessários para o agente text-to-SQL
import os
import re
from typing import Dict, List, Any, Optional

# Databricks SDK e Foundation Models
from databricks import agents
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# MLflow para logging e deployment
import mlflow
mlflow.set_registry_uri("databricks-uc")
import mlflow.deployments

# LangChain core (apenas o necessário)
from langchain_core.messages import HumanMessage, SystemMessage

# PySpark para execução de queries
from pyspark.sql import SparkSession

##Configuração Inicial

Vamos configurar o modelo Foundation e definir o contexto de schema que o agente vai usar.

In [0]:
# Configuração do Foundation Model
# Usando modelo disponível no workspace
MODEL_NAME = "databricks-llama-4-maverick"  # Llama 4 - 17B active params, 128K context

# Criar cliente MLflow para acessar Foundation Models
deployment_client = mlflow.deployments.get_deploy_client("databricks")

# Definir o schema baseado nas tabelas reais do Unity Catalog
SCHEMA_CONTEXT = """
## Schema do Banco de Dados - Dados de Táxi

Você tem acesso às seguintes tabelas no Unity Catalog:

### Tabela: gold.taxidata.yellow_taxi_data
Descrição: Dados de corridas de táxi amarelo (Yellow Taxi)
Colunas:
- id_vendor (LONG): ID do fornecedor/empresa de táxi
- dh_inicio_corrida (TIMESTAMP_NTZ): Data e hora de início da corrida
- dh_final_corrida (TIMESTAMP_NTZ): Data e hora de término da corrida
- vl_total_corrida (DOUBLE): Valor total cobrado pela corrida em dólares
- nr_passageiros (DOUBLE): Número de passageiros na corrida

### Tabela: gold.taxidata.green_taxi_data
Descrição: Dados de corridas de táxi verde (Green Taxi)
Colunas:
- id_vendor (LONG): ID do fornecedor/empresa de táxi
- dh_inicio_corrida (TIMESTAMP_NTZ): Data e hora de início da corrida
- dh_final_corrida (TIMESTAMP_NTZ): Data e hora de término da corrida
- vl_total_corrida (DOUBLE): Valor total cobrado pela corrida em dólares
- nr_passageiros (DOUBLE): Número de passageiros na corrida

## Notas importantes:
- Ambas as tabelas têm a mesma estrutura
- Use TIMESTAMP_NTZ para filtros de data/hora (não precisa de timezone)
- Para calcular duração da corrida: dh_final_corrida - dh_inicio_corrida
- Os valores monetários estão em dólares americanos (USD)
- Para análises combinadas, você pode usar UNION ALL entre as duas tabelas
"""

print("Configuração concluída")
print(f"Modelo: {MODEL_NAME}")

In [0]:
class TextToSQLAgent:
    """
    Agente avançado de text-to-SQL com validação, execução e tratamento de erros.
    """
    
    def __init__(self, model_name: str, schema_context: str):
        """
        Inicializa o agente.
        
        Args:
            model_name: Nome do Foundation Model a usar
            schema_context: Contexto do schema do banco de dados
        """
        self.model_name = model_name
        self.schema_context = schema_context
        self.deployment_client = mlflow.deployments.get_deploy_client("databricks")
        self.spark = SparkSession.builder.getOrCreate()
        self.query_history = []  # Histórico de queries para aprendizado
    
    def generate_sql(self, question: str, attempt: int = 1) -> Dict[str, Any]:
        """
        Gera SQL a partir de uma pergunta em linguagem natural.
        
        Args:
            question: Pergunta em linguagem natural
            attempt: Número da tentativa (para retry com contexto)
        
        Returns:
            Dict com 'sql', 'confidence' e 'explanation'
        """
        
        # Adicionar contexto de erros anteriores se for retry
        error_context = ""
        if attempt > 1 and self.query_history:
            last_error = self.query_history[-1].get('error')
            if last_error:
                error_context = f"\n\n Tentativa anterior falhou com erro: {last_error}\nPor favor, corrija o SQL."
        
        system_prompt = f"""
        Você é um especialista em SQL que converte perguntas em linguagem natural para queries SQL válidas.
        
        {self.schema_context}
        
        ## Regras importantes:
        
        1. Gere APENAS o código SQL, sem explicações ou markdown
        2. Use nomes de tabelas totalmente qualificados (catalog.schema.table)
        3. Para datas:
           - Use CURRENT_DATE() para data atual
           - Use DATE_SUB(CURRENT_DATE(), N) para N dias atrás
           - Use MONTH(), YEAR() para extrair partes de data
        4. Para agregações, sempre use GROUP BY
        5. Use LIMIT 100 por padrão para evitar resultados muito grandes
        6. Valide que todas as colunas existem no schema
        7. Use JOINs apropriados quando relacionar tabelas
        8. Para "último mês", use os últimos 30 dias
        9. Use aliases descritivos em português
        10. Você pode usar CTEs (WITH) para queries complexas
        
        {error_context}
        """
        
        # Enviar pergunta para o modelo usando MLflow deployments
        response = self.deployment_client.predict(
            endpoint=self.model_name,
            inputs={
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Pergunta: {question}"}
                ],
                "temperature": 0.1,
                "max_tokens": 2000
            }
        )
        
        # Extrair o conteúdo da resposta
        sql_query = response['choices'][0]['message']['content'].strip()
        
        # Limpar markdown
        sql_query = re.sub(r'^```sql\s*', '', sql_query)
        sql_query = re.sub(r'\s*```$', '', sql_query)
        sql_query = sql_query.strip()
        
        return {
            'sql': sql_query,
            'confidence': 'high' if attempt == 1 else 'medium',
            'attempt': attempt
        }
    
    def validate_sql(self, sql: str) -> Dict[str, Any]:
        """
        Valida o SQL antes de executar.
        
        Args:
            sql: Query SQL a validar
        
        Returns:
            Dict com 'valid' (bool) e 'error' (str ou None)
        """
        try:
            # Validações básicas
            sql_upper = sql.upper().strip()
            
            # Bloquear comandos perigosos
            dangerous_keywords = ['DROP', 'DELETE', 'TRUNCATE', 'ALTER', 'CREATE', 'INSERT', 'UPDATE']
            for keyword in dangerous_keywords:
                if keyword in sql_upper:
                    return {
                        'valid': False,
                        'error': f"Comando {keyword} não é permitido. Apenas queries SELECT são aceitas."
                    }
            
            # Verificar se é um SELECT ou WITH (CTEs são válidas)
            if not (sql_upper.startswith('SELECT') or sql_upper.startswith('WITH')):
                return {
                    'valid': False,
                    'error': "A query deve começar com SELECT ou WITH (para CTEs)."
                }
            
            # Validar sintaxe com explain
            try:
                self.spark.sql(f"EXPLAIN {sql}")
            except Exception as e:
                return {
                    'valid': False,
                    'error': f"Erro de sintaxe SQL: {str(e)}"
                }
            
            return {'valid': True, 'error': None}
        
        except Exception as e:
            return {'valid': False, 'error': str(e)}
    
    def execute_sql(self, sql: str) -> Dict[str, Any]:
        """
        Executa a query SQL e retorna os resultados.
        
        Args:
            sql: Query SQL a executar
        
        Returns:
            Dict com 'success', 'data', 'row_count', 'error'
        """
        try:
            df = self.spark.sql(sql)
            result_df = df.limit(1000)  # Limite de segurança
            pandas_df = result_df.toPandas()
            
            return {
                'success': True,
                'data': pandas_df,
                'row_count': len(pandas_df),
                'error': None
            }
        
        except Exception as e:
            return {
                'success': False,
                'data': None,
                'row_count': 0,
                'error': str(e)
            }
    
    def answer_question(self, question: str, max_retries: int = 3) -> Dict[str, Any]:
        """
        Responde uma pergunta: gera SQL, valida, executa e formata resultado.
        
        Args:
            question: Pergunta em linguagem natural
            max_retries: Número máximo de tentativas em caso de erro
        
        Returns:
            Dict com todos os detalhes da resposta
        """
        print(f"\n Pergunta: {question}\n")
        
        for attempt in range(1, max_retries + 1):
            # Gerar SQL
            print(f" Tentativa {attempt}: Gerando SQL...")
            sql_result = self.generate_sql(question, attempt)
            sql = sql_result['sql']
            
            print(f"\n SQL Gerado:\n{sql}\n")
            
            # Validar SQL
            print(" Validando SQL...")
            validation = self.validate_sql(sql)
            
            if not validation['valid']:
                print(f" Validação falhou: {validation['error']}")
                self.query_history.append({
                    'question': question,
                    'sql': sql,
                    'error': validation['error'],
                    'attempt': attempt
                })
                if attempt < max_retries:
                    print(" Tentando novamente...\n")
                    continue
                else:
                    return {
                        'success': False,
                        'question': question,
                        'sql': sql,
                        'error': validation['error'],
                        'data': None
                    }
            
            # Executar SQL
            print(" Executando query...")
            execution = self.execute_sql(sql)
            
            if not execution['success']:
                print(f" Execução falhou: {execution['error']}")
                self.query_history.append({
                    'question': question,
                    'sql': sql,
                    'error': execution['error'],
                    'attempt': attempt
                })
                if attempt < max_retries:
                    print(" Tentando novamente...\n")
                    continue
                else:
                    return {
                        'success': False,
                        'question': question,
                        'sql': sql,
                        'error': execution['error'],
                        'data': None
                    }
            
            # Sucesso!
            print(f" Query executada com sucesso! {execution['row_count']} linhas retornadas.\n")
            
            result = {
                'success': True,
                'question': question,
                'sql': sql,
                'data': execution['data'],
                'row_count': execution['row_count'],
                'error': None,
                'attempt': attempt
            }
            
            self.query_history.append(result)
            return result
        
        # Se chegou aqui, todas as tentativas falharam
        return {
            'success': False,
            'question': question,
            'sql': None,
            'error': 'Todas as tentativas falharam',
            'data': None
        }

print("Classe TextToSQLAgent criada com sucesso")

In [0]:
# Criar instância do agente
agent = TextToSQLAgent(
    model_name=MODEL_NAME,
    schema_context=SCHEMA_CONTEXT
)

print("Agente text-to-SQL inicializado!")
print(f"Modelo: {MODEL_NAME}")
print(f"Schema: 2 tabelas de táxi configuradas (yellow e green)")
print("\nPronto para responder perguntas!")

## Exemplos Práticos

Vamos testar o agente com perguntas comuns sobre os dados de táxi.

**Nota**: As queries serão geradas e executadas contra as tabelas [gold.taxidata.yellow_taxi_data](#table/gold/taxidata/yellow_taxi_data) e [gold.taxidata.green_taxi_data](#table/gold/taxidata/green_taxi_data).

In [0]:
# Exemplo 1: Total de corridas no mês de abril

result1 = agent.answer_question("Quantas corridas de táxi amarelo tivemos nos 30 dias de abril em 2023?")

if result1['success']:
    display(result1['data'])
else:
    print(f"⚠️ Erro: {result1['error']}")

In [0]:
# Exemplo 2: Receita total por tipo de táxi

result2 = agent.answer_question("Qual foi a receita total de corridas de táxi verde vs amarelo?")

if result2['success']:
    display(result2['data'])
else:
    print(f"⚠️ Erro: {result2['error']}")

In [0]:
# Exemplo 3: Duração média das corridas

result3 = agent.answer_question("Qual é a duração média das corridas de táxi em minutos?")

if result3['success']:
    display(result3['data'])
else:
    print(f"⚠️ Erro: {result3['error']}")

In [0]:
# Exemplo 4: Corridas por hora do dia

result4 = agent.answer_question("Quantas corridas acontecem por hora do dia em média?")

if result4['success']:
    display(result4['data'])
else:
    print(f"⚠️ Erro: {result4['error']}")